# Éco-conception avec CodeCarbon — ML & DL

**Partie 2** de la démarche d'optimisation raisonnée (fait suite au notebook Optuna/XGBoost). Objectif : mesurer, pas supposer, le coût énergétique et carbone de l'entraînement, et l'utiliser comme critère de décision au même titre que la performance.

1. **Instrumentation** : `EmissionsTracker` autour de l'entraînement ML (baseline XGBoost) et DL (autoencodeur SSIM) — mêmes wrapper et conventions pour les deux.
2. **Étude lourde vs étude frugale** : deux stratégies Optuna comparées sur performance **et** empreinte carbone, coût par point de PR-AUC gagné.
3. Réponses aux questions : gain proportionné ? quand la parcimonie est-elle la bonne décision Green AI ?

Tous les fichiers de sortie (`emissions.csv`, tableaux, graphiques) sont écrits dans `artifacts/ingestions/output/`.

In [ ]:
import time
_notebook_start_time = time.time()

import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import xgboost as xgb
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner, NopPruner
from sklearn.model_selection import TimeSeriesSplit, train_test_split
from sklearn.metrics import average_precision_score, roc_auc_score

from codecarbon import EmissionsTracker

SEED = 42
np.random.seed(SEED)
optuna.logging.set_verbosity(optuna.logging.WARNING)

OUTPUT_DIR = "artifacts/ingestions/output"   # <- chemin relatif à la racine du projet
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Intensité carbone forcée plutôt que géolocalisation automatique (observée peu fiable/reproductible
# d'une exécution à l'autre dans cet environnement) — ordre de grandeur du mix électrique français.
FRANCE_GRID_INTENSITY = 56.0  # gCO2eq/kWh (source ADEME/RTE, mix français très largement décarboné)

def make_tracker(project_name):
    return EmissionsTracker(
        project_name=project_name,
        output_dir=OUTPUT_DIR,
        output_file="emissions.csv",
        force_carbon_intensity_g_co2e_kwh=FRANCE_GRID_INTENSITY,
        log_level="error",
        measure_power_secs=1,
        allow_multiple_runs=True,
    )

steps_log = []  # accumule : étape, durée (s), kWh, gCO2eq, perf obtenue, détail

In [ ]:
def get_last_energy_kwh(project_name):
    """Relit emissions.csv pour récupérer le kWh authentique de la dernière ligne d'un projet donné."""
    emissions_df = pd.read_csv(os.path.join(OUTPUT_DIR, "emissions.csv"))
    return emissions_df[emissions_df.project_name == project_name].iloc[-1]["energy_consumed"]

## 1. Instrumenter l'entraînement

### 1.1 Chargement des données ML (`gold_dataset`)

In [ ]:
GOLD_PATH = "artifacts/ingestions/datas/gold_dataset.parquet"  # <- chemin relatif à la racine du projet

EXCLUDE_BASE = [
    'machine_id_std', 'window_start', 'window_end',
    'future_incident_count_6h', 'label_failure_next_6h',
    'future_incident_count_12h', 'label_failure_next_12h',
    'future_incident_count_24h', 'label_failure_next_24h',
    'future_incident_count_48h', 'label_failure_next_48h',
    'incident_count_1h', 'incident_max_severity_1h',
    'incident_count_prev_24h', 'incident_max_severity_prev_24h',
    'incident_count_prev_7d', 'hours_since_last_incident',
    'type_surchauffe_count_prev_24h', 'type_baisse_pression_count_prev_24h',
    'type_vibration_count_prev_24h', 'type_bruit_mecanique_count_prev_24h',
    'type_surconsommation_count_prev_24h', 'type_blocage_mecanique_count_prev_24h',
    'type_alarme_capteur_count_prev_24h', 'type_arret_urgence_count_prev_24h',
    'type_defaut_qualite_count_prev_24h',
    'days_since_last_maintenance', 'maintenance_count_prev_30d',
    'split_set',
]
HORIZON = 'label_failure_next_24h'
ALL_HORIZONS = ['label_failure_next_6h', 'label_failure_next_12h', 'label_failure_next_24h', 'label_failure_next_48h']

gold_df = pd.read_parquet(GOLD_PATH)
feats = [c for c in gold_df.columns if c not in EXCLUDE_BASE and c not in ALL_HORIZONS]

train_df = gold_df[gold_df.split_set == 'train'].sort_values('window_start').reset_index(drop=True)
val_df = gold_df[gold_df.split_set == 'validation']

X_train, y_train = train_df[feats], train_df[HORIZON].astype(int)
X_val, y_val = val_df[feats], val_df[HORIZON].astype(int)

print(f"Train : {len(X_train)}  ({y_train.mean():.1%} positifs)  —  Validation : {len(X_val)}")

### 1.2 Entraînement ML instrumenté (baseline XGBoost)

In [ ]:
BASELINE_PARAMS = dict(
    n_estimators=300, max_depth=5, learning_rate=0.1,
    subsample=0.9, colsample_bytree=0.9, min_child_weight=1,
    reg_lambda=1.0, reg_alpha=0.0,
)

tracker = make_tracker("ml_baseline_xgboost")
tracker.start()
_t0 = time.time()

baseline_model = xgb.XGBClassifier(
    **BASELINE_PARAMS, tree_method="hist", n_jobs=1, eval_metric="aucpr", random_state=SEED,
)
baseline_model.fit(X_train, y_train)

_duration_ml = time.time() - _t0
_emissions_kg_ml = tracker.stop()

val_proba_baseline = baseline_model.predict_proba(X_val)[:, 1]
baseline_prauc = average_precision_score(y_val, val_proba_baseline)
baseline_auc = roc_auc_score(y_val, val_proba_baseline)

_kwh_ml = get_last_energy_kwh("ml_baseline_xgboost")
steps_log.append({
    "étape": "ML — baseline XGBoost", "durée_s": _duration_ml,
    "kWh": _kwh_ml, "gCO2eq": _emissions_kg_ml * 1000,
    "perf_obtenue": baseline_prauc, "perf_metric": "PR-AUC (val)",
})

print(f"ML baseline — durée={_duration_ml:.1f}s  émissions={_emissions_kg_ml*1000:.4f} gCO2eq  PR-AUC(val)={baseline_prauc:.4f}")

### 1.3 Entraînement DL instrumenté (autoencodeur convolutionnel, perte SSIM)

Même wrapper `EmissionsTracker` (`make_tracker`), même fichier `emissions.csv` — pour comparer directement le coût ML vs DL dans le même tableau. Dataset `bottle` (MVTec-AD), architecture et perte SSIM déjà validées dans les travaux précédents.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Model
import cv2
from PIL import Image
import albumentations as A
from pathlib import Path

tf.random.set_seed(SEED)

BOTTLE_DIR = Path("data/bottle")   # <- adapter au chemin réel du dataset bottle
IMG_SIZE = 128
DL_BATCH_SIZE = 16
DL_VAL_FRACTION = 0.15
DL_EPOCHS = 30

def load_and_resize_image(path, size=IMG_SIZE):
    img = Image.open(path).convert("RGB")
    img = np.array(img)
    return cv2.resize(img, (size, size), interpolation=cv2.INTER_AREA)

def normalize(img):
    return img.astype(np.float32) / 255.0

train_good = sorted((BOTTLE_DIR / "train" / "good").glob("*.png"))
dl_train_paths, dl_val_paths = train_test_split(train_good, test_size=DL_VAL_FRACTION, random_state=SEED)

dl_train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.Rotate(limit=15, p=0.5, border_mode=cv2.BORDER_REPLICATE),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=0, border_mode=cv2.BORDER_REPLICATE, p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.5),
])

def make_loader(augment):
    def _load(path):
        path = path.numpy().decode("utf-8")
        img = load_and_resize_image(path)
        if augment:
            img = dl_train_transform(image=img)["image"]
        return normalize(img).astype(np.float32)
    def _tf_wrapper(path):
        img = tf.py_function(_load, [path], tf.float32)
        img.set_shape((IMG_SIZE, IMG_SIZE, 3))
        return img
    return _tf_wrapper

def build_dl_dataset(paths, augment=False, shuffle=False, batch_size=DL_BATCH_SIZE):
    ds = tf.data.Dataset.from_tensor_slices([str(p) for p in paths])
    if shuffle:
        ds = ds.shuffle(buffer_size=len(paths), seed=SEED)
    ds = ds.map(make_loader(augment), num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

dl_train_ds = build_dl_dataset(dl_train_paths, augment=True, shuffle=True)
dl_val_ds = build_dl_dataset(dl_val_paths, augment=False, shuffle=False)
dl_train_ds_xy = dl_train_ds.map(lambda x: (x, x))
dl_val_ds_xy = dl_val_ds.map(lambda x: (x, x))

print(f"DL — train: {len(dl_train_paths)}  val: {len(dl_val_paths)}")

In [ ]:
def build_autoencoder(img_size=IMG_SIZE, base_filters=32):
    inputs = layers.Input(shape=(img_size, img_size, 3), name="input_image")
    x = layers.Conv2D(base_filters, 3, strides=2, padding="same", activation="relu")(inputs)
    x = layers.Conv2D(base_filters * 2, 3, strides=2, padding="same", activation="relu")(x)
    x = layers.Conv2D(base_filters * 4, 3, strides=2, padding="same", activation="relu")(x)
    x = layers.Conv2D(base_filters * 8, 3, strides=2, padding="same", activation="relu")(x)
    x = layers.Conv2DTranspose(base_filters * 4, 3, strides=2, padding="same", activation="relu")(x)
    x = layers.Conv2DTranspose(base_filters * 2, 3, strides=2, padding="same", activation="relu")(x)
    x = layers.Conv2DTranspose(base_filters, 3, strides=2, padding="same", activation="relu")(x)
    outputs = layers.Conv2DTranspose(3, 3, strides=2, padding="same", activation="sigmoid")(x)
    return Model(inputs, outputs, name="conv_autoencoder")


def ssim_loss(y_true, y_pred):
    return 1.0 - tf.reduce_mean(tf.image.ssim(y_true, y_pred, max_val=1.0))


def mse_metric(y_true, y_pred):
    return tf.reduce_mean(tf.square(y_true - y_pred))
mse_metric.__name__ = "mse"


dl_model = build_autoencoder()
dl_model.compile(optimizer="adam", loss=ssim_loss, metrics=[mse_metric])
early_stopping_dl = tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)

tracker = make_tracker("dl_autoencoder_ssim")
tracker.start()
_t0 = time.time()

dl_history = dl_model.fit(
    dl_train_ds_xy, validation_data=dl_val_ds_xy,
    epochs=DL_EPOCHS, callbacks=[early_stopping_dl], verbose=0,
)

_duration_dl = time.time() - _t0
_emissions_kg_dl = tracker.stop()

dl_val_ssim_final = 1.0 - dl_history.history["val_loss"][-1]

_kwh_dl = get_last_energy_kwh("dl_autoencoder_ssim")
steps_log.append({
    "étape": "DL — autoencodeur SSIM", "durée_s": _duration_dl,
    "kWh": _kwh_dl, "gCO2eq": _emissions_kg_dl * 1000,
    "perf_obtenue": dl_val_ssim_final, "perf_metric": "SSIM (val)",
})

print(f"DL autoencodeur — durée={_duration_dl:.1f}s  émissions={_emissions_kg_dl*1000:.4f} gCO2eq  "
      f"SSIM(val)={dl_val_ssim_final:.4f}  ({len(dl_history.history['loss'])} epochs)")

### 1.4 Rendu — tableau `étape → durée, kWh, gCO2eq, perf obtenue`

In [ ]:
steps_table = pd.DataFrame(steps_log)
steps_table.to_csv(os.path.join(OUTPUT_DIR, "steps_summary.csv"), index=False)
steps_table

### 1.5 Graphiques pertinents

Deux lectures complémentaires : (1) répartition kWh/gCO2eq par étape (échelles très différentes entre ML tabulaire et DL image — barre logarithmique), (2) rapport performance/coût pour situer chaque étape.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].bar(steps_table["étape"], steps_table["gCO2eq"], color=["#4C72B0", "#DD8452"])
axes[0].set_ylabel("gCO2eq (échelle log)")
axes[0].set_yscale("log")
axes[0].set_title("Émissions par étape")
axes[0].tick_params(axis="x", rotation=15)

axes[1].bar(steps_table["étape"], steps_table["durée_s"], color=["#4C72B0", "#DD8452"])
axes[1].set_ylabel("Durée (s)")
axes[1].set_title("Durée par étape")
axes[1].tick_params(axis="x", rotation=15)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "emissions_par_etape.png"), dpi=130)
plt.show()

## 2. Étude lourde vs étude frugale

Deux stratégies Optuna, sur le même problème ML (XGBoost, horizon 24h), instrumentées de la même façon :

| | Étude **lourde** | Étude **frugale** |
|---|---|---|
| Espace de recherche | Large (bornes ~2-3x plus larges) | Resserré (bornes "raisonnées", cf. notebook précédent) |
| Essais | 40 | 15 |
| Pruning | Aucun (`NopPruner`) | Agressif (`MedianPruner`, coupe dès le 1er fold) |
| Folds CV | 3, données complètes | 2, sous-échantillon 50% |

Optimisation temps de traitement appliquée aux deux (cf. notebook précédent) : `tree_method="hist"`, `early_stopping_rounds`, `n_estimators` borné par l'early stopping plutôt que fixé arbitrairement haut.

In [ ]:
X_train_full = X_train.values
y_train_full = y_train.values
neg_full, pos_full = (y_train_full == 0).sum(), (y_train_full == 1).sum()
RATIO_FULL = neg_full / pos_full

# Sous-échantillon pour la recherche frugale uniquement (stride systématique, préserve l'ordre temporel)
X_train_frugal = X_train_full[::2]
y_train_frugal = y_train_full[::2]
neg_frugal, pos_frugal = (y_train_frugal == 0).sum(), (y_train_frugal == 1).sum()
RATIO_FRUGAL = neg_frugal / pos_frugal

print(f"Train complet (lourd) : {len(X_train_full)} lignes")
print(f"Train sous-échantillonné (frugal) : {len(X_train_frugal)} lignes")

### 2.1 Étude lourde — espace large, 40 essais, sans pruning

In [ ]:
def objective_heavy(trial: optuna.Trial) -> float:
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 800),
        "max_depth": trial.suggest_int("max_depth", 2, 12),
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.5, log=True),
        "subsample": trial.suggest_float("subsample", 0.4, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.4, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-4, 100.0, log=True),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 100.0, log=True),
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 1.0, RATIO_FULL * 2),
    }
    tscv = TimeSeriesSplit(n_splits=3)
    scores = []
    for tr_idx, va_idx in tscv.split(X_train_full):
        model = xgb.XGBClassifier(
            **params, tree_method="hist", n_jobs=1, eval_metric="aucpr",
            early_stopping_rounds=20, random_state=SEED,
        )
        model.fit(X_train_full[tr_idx], y_train_full[tr_idx],
                  eval_set=[(X_train_full[va_idx], y_train_full[va_idx])], verbose=False)
        proba = model.predict_proba(X_train_full[va_idx])[:, 1]
        scores.append(average_precision_score(y_train_full[va_idx], proba))
        # Pas de pruning ici (NopPruner) — chaque essai va jusqu'au bout des 3 folds
    return float(np.mean(scores))


N_TRIALS_HEAVY = 40

tracker = make_tracker("study_heavy")
tracker.start()
_t0 = time.time()

study_heavy = optuna.create_study(direction="maximize", sampler=TPESampler(seed=SEED), pruner=NopPruner())
study_heavy.optimize(objective_heavy, n_trials=N_TRIALS_HEAVY)

_duration_heavy = time.time() - _t0
_emissions_kg_heavy = tracker.stop()
_kwh_heavy = get_last_energy_kwh("study_heavy")

print(f"Étude lourde — {N_TRIALS_HEAVY} essais — durée={_duration_heavy:.1f}s ({_duration_heavy/60:.1f} min)  "
      f"émissions={_emissions_kg_heavy*1000:.4f} gCO2eq  meilleur PR-AUC(CV)={study_heavy.best_value:.4f}")

### 2.2 Étude frugale — espace resserré, 15 essais, pruning agressif

In [ ]:
def objective_frugal(trial: optuna.Trial) -> float:
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 80, 200),
        "max_depth": trial.suggest_int("max_depth", 3, 6),
        "learning_rate": trial.suggest_float("learning_rate", 0.03, 0.2, log=True),
        "subsample": trial.suggest_float("subsample", 0.7, 0.95),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.7, 0.95),
        "min_child_weight": trial.suggest_int("min_child_weight", 2, 8),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.01, 3.0, log=True),
        "reg_alpha": trial.suggest_float("reg_alpha", 0.01, 3.0, log=True),
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 1.0, RATIO_FRUGAL),
    }
    tscv = TimeSeriesSplit(n_splits=2)
    scores = []
    for fold_i, (tr_idx, va_idx) in enumerate(tscv.split(X_train_frugal)):
        model = xgb.XGBClassifier(
            **params, tree_method="hist", n_jobs=1, eval_metric="aucpr",
            early_stopping_rounds=15, random_state=SEED,
        )
        model.fit(X_train_frugal[tr_idx], y_train_frugal[tr_idx],
                  eval_set=[(X_train_frugal[va_idx], y_train_frugal[va_idx])], verbose=False)
        proba = model.predict_proba(X_train_frugal[va_idx])[:, 1]
        scores.append(average_precision_score(y_train_frugal[va_idx], proba))
        # Pruning agressif : dès le 1er fold, un essai clairement mauvais est coupé
        trial.report(float(np.mean(scores)), step=fold_i)
        if trial.should_prune():
            raise optuna.TrialPruned()
    return float(np.mean(scores))


N_TRIALS_FRUGAL = 15

tracker = make_tracker("study_frugal")
tracker.start()
_t0 = time.time()

study_frugal = optuna.create_study(
    direction="maximize", sampler=TPESampler(seed=SEED),
    pruner=MedianPruner(n_warmup_steps=0, n_startup_trials=2),
)
study_frugal.optimize(objective_frugal, n_trials=N_TRIALS_FRUGAL)

_duration_frugal = time.time() - _t0
_emissions_kg_frugal = tracker.stop()
_kwh_frugal = get_last_energy_kwh("study_frugal")

n_pruned_frugal = len([t for t in study_frugal.trials if t.state == optuna.trial.TrialState.PRUNED])

print(f"Étude frugale — {N_TRIALS_FRUGAL} essais ({n_pruned_frugal} élagués) — durée={_duration_frugal:.1f}s "
      f"({_duration_frugal/60:.1f} min)  émissions={_emissions_kg_frugal*1000:.4f} gCO2eq  "
      f"meilleur PR-AUC(CV)={study_frugal.best_value:.4f}")

### 2.3 Évaluation à armes égales sur la validation

Les scores CV ci-dessus ne sont pas directement comparables à la baseline (protocoles différents : espace complet vs sous-échantillon, 3 vs 2 folds). On réentraîne chaque meilleur essai sur `X_train` complet et on évalue sur `X_val` — même protocole que la baseline (étape 1.2).

In [ ]:
def retrain_and_eval(best_params, run_name):
    tracker = make_tracker(run_name)
    tracker.start()
    _t0 = time.time()
    model = xgb.XGBClassifier(**best_params, tree_method="hist", n_jobs=1, eval_metric="aucpr", random_state=SEED)
    model.fit(X_train, y_train)
    duration = time.time() - _t0
    emissions_kg = tracker.stop()
    kwh = get_last_energy_kwh(run_name)
    proba = model.predict_proba(X_val)[:, 1]
    prauc = average_precision_score(y_val, proba)
    return {"duration": duration, "emissions_kg": emissions_kg, "kwh": kwh, "prauc_val": prauc}


retrain_heavy = retrain_and_eval(study_heavy.best_params, "retrain_heavy_best")
retrain_frugal = retrain_and_eval(study_frugal.best_params, "retrain_frugal_best")

print(f"Lourd  — PR-AUC(val) réentraîné = {retrain_heavy['prauc_val']:.4f}")
print(f"Frugal — PR-AUC(val) réentraîné = {retrain_frugal['prauc_val']:.4f}")
print(f"Baseline (rappel)     = {baseline_prauc:.4f}")

### 2.4 Coût par point de PR-AUC gagné

In [ ]:
# Coût total = recherche (étude complète) + réentraînement final, par stratégie
total_duration_heavy = _duration_heavy + retrain_heavy["duration"]
total_gco2_heavy = _emissions_kg_heavy * 1000 + retrain_heavy["emissions_kg"] * 1000
total_kwh_heavy = _kwh_heavy + retrain_heavy["kwh"]
gain_heavy = retrain_heavy["prauc_val"] - baseline_prauc

total_duration_frugal = _duration_frugal + retrain_frugal["duration"]
total_gco2_frugal = _emissions_kg_frugal * 1000 + retrain_frugal["emissions_kg"] * 1000
total_kwh_frugal = _kwh_frugal + retrain_frugal["kwh"]
gain_frugal = retrain_frugal["prauc_val"] - baseline_prauc


def cost_per_point(total_value, gain, unit):
    if gain <= 0:
        return f"non défini (aucun gain — {unit} dépensé pour rien ou une régression)"
    return f"{total_value / gain:.4f} {unit} par point de PR-AUC"


comparison = pd.DataFrame([
    {"stratégie": "Baseline", "essais": 1, "durée_s": _duration_ml, "gCO2eq": _emissions_kg_ml * 1000,
     "kWh": _kwh_ml, "PR-AUC (val)": baseline_prauc, "gain vs baseline": 0.0},
    {"stratégie": "Frugale", "essais": N_TRIALS_FRUGAL, "durée_s": total_duration_frugal, "gCO2eq": total_gco2_frugal,
     "kWh": total_kwh_frugal, "PR-AUC (val)": retrain_frugal["prauc_val"], "gain vs baseline": gain_frugal},
    {"stratégie": "Lourde", "essais": N_TRIALS_HEAVY, "durée_s": total_duration_heavy, "gCO2eq": total_gco2_heavy,
     "kWh": total_kwh_heavy, "PR-AUC (val)": retrain_heavy["prauc_val"], "gain vs baseline": gain_heavy},
])
comparison["gCO2eq par point PR-AUC"] = comparison.apply(
    lambda r: r["gCO2eq"] / r["gain vs baseline"] if r["gain vs baseline"] > 0 else np.nan, axis=1)
comparison["kWh par point PR-AUC"] = comparison.apply(
    lambda r: r["kWh"] / r["gain vs baseline"] if r["gain vs baseline"] > 0 else np.nan, axis=1)

comparison.to_csv(os.path.join(OUTPUT_DIR, "comparaison_lourd_vs_frugal.csv"), index=False)
comparison

### 2.5 Rendu — performance vs CO2

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5.5))

colors = {"Baseline": "#55A868", "Frugale": "#DD8452", "Lourde": "#C44E52"}
for _, row in comparison.iterrows():
    ax.scatter(row["gCO2eq"], row["PR-AUC (val)"], s=180, color=colors[row["stratégie"]], label=row["stratégie"], zorder=3)
    ax.annotate(row["stratégie"], (row["gCO2eq"], row["PR-AUC (val)"]),
                textcoords="offset points", xytext=(8, 6), fontsize=10)

ax.set_xscale("log")
ax.set_xlabel("Émissions (gCO2eq, échelle log)")
ax.set_ylabel("PR-AUC (validation)")
ax.set_title("Performance vs empreinte carbone — baseline, frugale, lourde")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "performance_vs_co2.png"), dpi=130)
plt.show()

### 2.6 Réponses

**L'étude lourde apporte-t-elle un gain proportionné à son coût ?**

Comparer directement `gain vs baseline` et `gCO2eq` des deux stratégies (tableau 2.4) : si l'étude lourde consomme (très grossièrement) `N_TRIALS_HEAVY / N_TRIALS_FRUGAL` fois plus d'essais et un espace de recherche plus large sans pruning, son coût total est structurellement plus élevé — la question est de savoir si son gain de PR-AUC l'est dans les mêmes proportions. Dans la plupart des cas pratiques sur ce type de problème (modèle déjà proche de son plafond avec des hyperparamètres raisonnables, cf. notebook précédent), **le gain marginal d'une recherche 2-3x plus coûteuse est généralement très inférieur au facteur de coût** — l'un des symptômes typiques de rendement décroissant en optimisation d'hyperparamètres. Regarder la colonne `gCO2eq par point PR-AUC` : si elle est nettement plus élevée pour la stratégie lourde que pour la frugale, c'est la confirmation chiffrée que le coût n'est pas proportionné au gain.

**Quand la parcimonie (modèle plus simple, budget réduit) est-elle la bonne décision Green AI ?**

Sur la base de cette comparaison, quelques repères concrets :

1. **Quand le gain de la stratégie lourde ne dépasse pas la variance inter-essais** observée dans le notebook précédent (étape 6, stabilité) — un gain qui n'est pas statistiquement distinguable du bruit ne justifie jamais un surcoût carbone, quel qu'il soit.
2. **Quand le `gCO2eq par point de PR-AUC` de la stratégie frugale est du même ordre de grandeur (ou meilleur) que celui de la stratégie lourde** — la parcimonie n'est alors pas un compromis, c'est simplement la meilleure décision sur les deux axes (performance et empreinte).
3. **Quand le contexte de déploiement ne valorise pas économiquement le dernier point de performance** — ex. un gain de +0.005 PR-AUC qui ne change pas la décision opérationnelle en aval (le seuil de rappel/FP reste dans la même zone, cf. étude de seuil du notebook DL) ne justifie pas un budget de calcul disproportionné.
4. **Quand l'itération rapide compte plus que l'optimum global** — en phase d'exploration (nouveau dataset, nouvelle feature), une étude frugale donne un signal directionnel utile en une fraction du temps/carbone, suffisant pour décider de la suite sans épuiser le budget sur un réglage fin prématuré.

À l'inverse, une étude lourde reste justifiable quand le gain de performance a une valeur métier explicite et quantifiée (ex. un point de rappel supplémentaire sur la détection de panne évite un coût de maintenance non planifiée bien supérieur au coût carbone de la recherche) — la parcimonie n'est pas une règle absolue, c'est un défaut à justifier de s'en écarter, pas l'inverse.

## 3. Temps de traitement global

Optimisations déjà appliquées dans ce notebook (mêmes principes que le notebook Optuna précédent) : `tree_method="hist"`, `early_stopping_rounds`, sous-échantillonnage pour la recherche frugale, pruning agressif, `n_estimators` borné par l'early stopping plutôt que fixé arbitrairement haut.

In [ ]:
_total_elapsed = time.time() - _notebook_start_time
print(f"Temps total d'exécution du notebook : {_total_elapsed:.1f} s  ({_total_elapsed/60:.1f} min)")
print(f"  dont ML baseline            : {_duration_ml:.1f} s")
print(f"  dont DL autoencodeur        : {_duration_dl:.1f} s")
print(f"  dont étude lourde ({N_TRIALS_HEAVY} essais)  : {_duration_heavy:.1f} s ({_duration_heavy/60:.1f} min)")
print(f"  dont étude frugale ({N_TRIALS_FRUGAL} essais) : {_duration_frugal:.1f} s ({_duration_frugal/60:.1f} min)")

print(f"\nTous les fichiers de sortie sont dans : {OUTPUT_DIR}/")
print("  - emissions.csv (détail CodeCarbon, une ligne par étape instrumentée)")
print("  - steps_summary.csv (tableau étape -> durée/kWh/gCO2eq/perf)")
print("  - comparaison_lourd_vs_frugal.csv")
print("  - emissions_par_etape.png")
print("  - performance_vs_co2.png")